# Universal XBRL Parser for Multi-Taxonomy Financial Filings

This notebook parses XBRL XML filings across multiple industry taxonomies such as:

- Commercial / Manufacturing / IT / Hospital / NBFC
- Banking
- Insurance

The parser is designed as a staged pipeline with the following phases:

1. Pre-processing and namespace normalization
2. Taxonomy detection and routing
3. Context selection for the correct reporting slice
4. Raw fact extraction with context awareness
5. Unified field mapping with fallback tag dictionaries
6. Validation and structured output generation

The goal of this notebook is to extract reliable, context-correct, and taxonomy-aware financial data from XBRL filings in a way that scales across different company types.

## 1. Import Required Libraries

In [1]:
import json
import re
import xml.etree.ElementTree as ET

from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict, Counter

## 2. Configure Input and Output Paths for Batch Parsing

In [2]:
INPUT_FOLDER = Path("../data/raw")
OUTPUT_FOLDER = Path("../data/processed")

PREFERRED_REPORT_TYPE = "Consolidated"
SAVE_ONE_JSON_PER_XML = True

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

xml_files = sorted(INPUT_FOLDER.glob("*.xml"))

print("Input folder:", INPUT_FOLDER.resolve())
print("Output folder:", OUTPUT_FOLDER.resolve())
print("Preferred report type:", PREFERRED_REPORT_TYPE)
print("XML files found:", len(xml_files))

for file_path in xml_files:
    print("-", file_path.name)

Input folder: C:\Users\Home\llmops-nse-rag\data\raw
Output folder: C:\Users\Home\llmops-nse-rag\data\processed
Preferred report type: Consolidated
XML files found: 9
- GI_117338_1350247_17012025074725.xml
- INDAS_118123_1365717_30012025030002.xml
- INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.xml
- LI_117242_1346294_15012025050606.xml
- NBFC_INDAS_118212_1366575_30012025073535.xml
- RELIANCE_1425445_25042025095716_WEB.xml
- RELIANCE_1487275_18072025081722_WEB.xml
- RELIANCE_1552809_17102025073643_WEB.xml
- RELIANCE_1602862_16012026073821_WEB.xml


## 3. Define a Helper to Load and Parse XML Safely

In [3]:
def load_xml_file(file_path: Path) -> tuple[ET.ElementTree, ET.Element]:
    if not file_path.exists():
        raise FileNotFoundError(f"XML file not found: {file_path}")

    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        return tree, root
    except ET.ParseError as e:
        raise ET.ParseError(f"Invalid XML file {file_path.name}: {e}")

## 4. Define a Helper to Strip XML Namespaces

In [4]:
def strip_namespaces(element: ET.Element) -> ET.Element:
    for elem in element.iter():
        if isinstance(elem.tag, str) and "}" in elem.tag:
            elem.tag = elem.tag.split("}", 1)[1]
    return element

## 5. Define a Helper to Detect Filing Taxonomy

In [5]:
def detect_taxonomy(root: ET.Element) -> str:
    header_text_parts = []

    for elem in root.iter():
        if isinstance(elem.tag, str):
            header_text_parts.append(elem.tag)

        for attr_key, attr_value in elem.attrib.items():
            header_text_parts.append(str(attr_key))
            header_text_parts.append(str(attr_value))

    header_text = " ".join(header_text_parts).lower()

    if "bank" in header_text or "banking" in header_text:
        return "BANK"

    if "insurance" in header_text or "generalinsurance" in header_text or "lifeinsurance" in header_text:
        return "INSURANCE"

    return "COMMERCIAL_NBFC"

## 6. Test XML Loading, Namespace Stripping, and Taxonomy Detection on One Sample File

In [6]:
if not xml_files:
    raise ValueError("No XML files found in the input folder.")

sample_file = xml_files[0]
tree, root = load_xml_file(sample_file)
root = strip_namespaces(root)
taxonomy = detect_taxonomy(root)

print("Sample file:", sample_file.name)
print("Clean root tag:", root.tag)
print("Detected taxonomy:", taxonomy)

Sample file: GI_117338_1350247_17012025074725.xml
Clean root tag: xbrl
Detected taxonomy: INSURANCE


## 7. Define a Helper to Build the Context Dictionary

In [7]:
def build_context_dictionary(root: ET.Element) -> Dict[str, Dict[str, Any]]:
    contexts = {}

    for ctx in root.findall("context"):
        context_id = ctx.get("id")
        if not context_id:
            continue

        context_data = {
            "id": context_id,
            "type": None,
            "start_date": None,
            "end_date": None,
            "instant_date": None,
            "dimension": None,
            "member": None,
        }

        period_node = ctx.find("period")
        if period_node is not None:
            start_date = period_node.findtext("startDate")
            end_date = period_node.findtext("endDate")
            instant_date = period_node.findtext("instant")

            if start_date and end_date:
                context_data["type"] = "duration"
                context_data["start_date"] = start_date
                context_data["end_date"] = end_date
            elif instant_date:
                context_data["type"] = "instant"
                context_data["instant_date"] = instant_date

        scenario_node = ctx.find("scenario")
        if scenario_node is not None:
            explicit_member = scenario_node.find(".//explicitMember")
            if explicit_member is not None:
                raw_dimension = explicit_member.get("dimension")
                raw_member = explicit_member.text

                context_data["dimension"] = (raw_dimension or "").split(":")[-1] or None
                context_data["member"] = (raw_member or "").split(":")[-1] or None

        contexts[context_id] = context_data

    return contexts

## 8. Inspect Context Structure on the Sample File

In [8]:
sample_contexts = build_context_dictionary(root)

print("Total contexts found:", len(sample_contexts))

for index, (context_id, context_data) in enumerate(sample_contexts.items()):
    if index >= 5:
        break
    print(f"\nContext ID: {context_id}")
    print(json.dumps(context_data, indent=2, ensure_ascii=False))

Total contexts found: 128

Context ID: OneD
{
  "id": "OneD",
  "type": "duration",
  "start_date": "2024-10-01",
  "end_date": "2024-12-31",
  "instant_date": null,
  "dimension": null,
  "member": null
}

Context ID: OneI
{
  "id": "OneI",
  "type": "instant",
  "start_date": null,
  "end_date": null,
  "instant_date": "2024-12-31",
  "dimension": null,
  "member": null
}

Context ID: FourD
{
  "id": "FourD",
  "type": "duration",
  "start_date": "2024-10-01",
  "end_date": "2024-12-31",
  "instant_date": null,
  "dimension": null,
  "member": null
}

Context ID: FourI
{
  "id": "FourI",
  "type": "instant",
  "start_date": null,
  "end_date": null,
  "instant_date": "2024-12-31",
  "dimension": null,
  "member": null
}

Context ID: OneOtherIncome01MemberD
{
  "id": "OneOtherIncome01MemberD",
  "type": "duration",
  "start_date": "2024-10-01",
  "end_date": "2024-12-31",
  "instant_date": null,
  "dimension": "DetailsOfOtherIncomeAxis",
  "member": "OtherIncome1Member"
}


## 9. Define a Helper to Detect Filing-Level Report Metadata

In [9]:
def extract_filing_metadata(root: ET.Element) -> Dict[str, Optional[str]]:
    metadata_candidates = {
        "company_name": [
            "NameOfTheCompany",
            "NameOfBank",
            "NameOfEntity",
        ],
        "report_type": [
            "NatureOfReportStandaloneConsolidated",
        ],
        "currency": [
            "DescriptionOfPresentationCurrency",
        ],
        "rounding": [
            "LevelOfRounding",
        ],
    }

    extracted_metadata = {
        "company_name": None,
        "report_type": None,
        "currency": None,
        "rounding": None,
    }

    all_text_by_tag = defaultdict(list)

    for elem in root.iter():
        tag_name = elem.tag if isinstance(elem.tag, str) else ""
        text_value = elem.text.strip() if elem.text and elem.text.strip() else None

        if tag_name and text_value:
            all_text_by_tag[tag_name].append(text_value)

    for target_field, candidate_tags in metadata_candidates.items():
        for candidate_tag in candidate_tags:
            values = all_text_by_tag.get(candidate_tag, [])
            if values:
                extracted_metadata[target_field] = values[0]
                break

    return extracted_metadata

## 10. Inspect Filing-Level Metadata on the Sample File

In [10]:
sample_metadata = extract_filing_metadata(root)

print("Sample filing metadata:")
print(json.dumps(sample_metadata, indent=2, ensure_ascii=False))

Sample filing metadata:
{
  "company_name": "ICICI Lombard General Insurance Company Limited",
  "report_type": "Standalone",
  "currency": "INR",
  "rounding": null
}


## 11. Define a Helper to Normalize Numeric Facts Using the `decimals` Attribute

In [11]:
def normalize_numeric_fact(raw_value: Optional[str], decimals: Optional[str]) -> Dict[str, Any]:
    if raw_value is None or str(raw_value).strip() == "":
        return {
            "raw_value": raw_value,
            "decimals": decimals,
            "normalized_value": None,
            "is_numeric": False,
        }

    raw_text = str(raw_value).replace(",", "").strip()

    try:
        numeric_value = float(raw_text)
    except ValueError:
        return {
            "raw_value": raw_value,
            "decimals": decimals,
            "normalized_value": raw_value,
            "is_numeric": False,
        }

    multiplier = 1

    if decimals == "-5":
        multiplier = 100_000
    elif decimals == "-7":
        multiplier = 10_000_000

    normalized_value = numeric_value * multiplier

    return {
        "raw_value": raw_value,
        "decimals": decimals,
        "normalized_value": normalized_value,
        "is_numeric": True,
    }

## 12. Define a Helper to Extract Raw Facts with Context Awareness

In [12]:
def extract_raw_facts(
    root: ET.Element,
    contexts: Dict[str, Dict[str, Any]],
    taxonomy: str,
    filing_metadata: Dict[str, Optional[str]],
) -> List[Dict[str, Any]]:
    raw_facts = []

    ignore_tags = {
        "context",
        "unit",
        "schemaRef",
        "identifier",
        "entity",
        "period",
        "startDate",
        "endDate",
        "instant",
        "scenario",
        "explicitMember",
        "typedMember",
    }

    for elem in root.iter():
        tag_name = elem.tag if isinstance(elem.tag, str) else ""
        if not tag_name or tag_name in ignore_tags:
            continue

        context_ref = elem.get("contextRef")
        if not context_ref or context_ref not in contexts:
            continue

        text_value = elem.text.strip() if elem.text and elem.text.strip() else None
        decimals = elem.get("decimals")

        numeric_info = normalize_numeric_fact(text_value, decimals)

        raw_fact = {
            "tag_name": tag_name,
            "text_value": text_value,
            "raw_value": numeric_info["raw_value"],
            "decimals": numeric_info["decimals"],
            "normalized_value": numeric_info["normalized_value"],
            "is_numeric": numeric_info["is_numeric"],
            "context_id": context_ref,
            "context": contexts[context_ref],
            "taxonomy": taxonomy,
            "filing_metadata": filing_metadata,
        }

        raw_facts.append(raw_fact)

    return raw_facts

## 13. Inspect Raw Facts on the Sample File

In [13]:
sample_raw_facts = extract_raw_facts(
    root=root,
    contexts=sample_contexts,
    taxonomy=taxonomy,
    filing_metadata=sample_metadata,
)

print("Total raw facts extracted:", len(sample_raw_facts))

for fact in sample_raw_facts[:5]:
    print(json.dumps(fact, indent=2, ensure_ascii=False))
    print("-" * 80)

Total raw facts extracted: 339
{
  "tag_name": "ScripCode",
  "text_value": "540716",
  "raw_value": "540716",
  "decimals": null,
  "normalized_value": 540716.0,
  "is_numeric": true,
  "context_id": "OneI",
  "context": {
    "id": "OneI",
    "type": "instant",
    "start_date": null,
    "end_date": null,
    "instant_date": "2024-12-31",
    "dimension": null,
    "member": null
  },
  "taxonomy": "INSURANCE",
  "filing_metadata": {
    "company_name": "ICICI Lombard General Insurance Company Limited",
    "report_type": "Standalone",
    "currency": "INR",
    "rounding": null
  }
}
--------------------------------------------------------------------------------
{
  "tag_name": "NSESymbol",
  "text_value": "ICICIGI",
  "raw_value": "ICICIGI",
  "decimals": null,
  "normalized_value": "ICICIGI",
  "is_numeric": false,
  "context_id": "OneI",
  "context": {
    "id": "OneI",
    "type": "instant",
    "start_date": null,
    "end_date": null,
    "instant_date": "2024-12-31",
    "

## 14. Define the Unified Financial Field Mapping Dictionary

In [14]:
UNIFIED_SCHEMA_MAPPING = {
    "Operating_Revenue": [
        "RevenueFromOperations",
        "InterestEarned",
        "GrossPremiumsWritten",
        "NetPremiumEarned",
    ],
    "Total_Income": [
        "Income",
        "OperatingIncome",
        "IncomeInShareholdersAccount",
    ],
    "Primary_Direct_Expense": [
        "CostOfMaterialsConsumed",
        "InterestExpended",
        "FinanceCosts",
        "IncurredClaims",
        "ClaimsPaid",
    ],
    "Employee_Expenses": [
        "EmployeeBenefitExpense",
        "EmployeesCost",
        "EmployeesRemunerationAndWelfareExpenses",
    ],
    "Net_Profit": [
        "ProfitLossForPeriod",
        "ProfitLossForThePeriod",
        "ProfitLossAfterTax",
    ],
    "Basic_EPS": [
        "BasicEarningsLossPerShareFromContinuingOperations",
        "BasicEarningsPerShareBeforeExtraordinaryItems",
        "BasicAndDilutedEPSBeforeExtraordinaryItemsNetOfTaxExpense",
    ],
}

## 15. Define a Helper to Map Raw Facts into the Unified Financial Schema

In [15]:
def map_facts_to_unified_schema(
    raw_facts: List[Dict[str, Any]],
    unified_mapping: Dict[str, List[str]],
) -> List[Dict[str, Any]]:
    facts_by_context = defaultdict(list)

    for fact in raw_facts:
        facts_by_context[fact["context_id"]].append(fact)

    unified_records = []

    for context_id, facts in facts_by_context.items():
        tag_lookup = defaultdict(list)

        for fact in facts:
            tag_lookup[fact["tag_name"]].append(fact)

        context_metadata = facts[0]["context"]
        taxonomy = facts[0]["taxonomy"]
        filing_metadata = facts[0]["filing_metadata"]

        unified_record = {
            "context_id": context_id,
            "context": context_metadata,
            "taxonomy": taxonomy,
            "filing_metadata": filing_metadata,
            "unified_fields": {},
        }

        for unified_field, candidate_tags in unified_mapping.items():
            selected_fact = None

            for candidate_tag in candidate_tags:
                candidate_facts = tag_lookup.get(candidate_tag, [])
                if candidate_facts:
                    selected_fact = candidate_facts[0]
                    break

            if selected_fact:
                unified_record["unified_fields"][unified_field] = {
                    "source_tag": selected_fact["tag_name"],
                    "value": selected_fact["normalized_value"] if selected_fact["is_numeric"] else selected_fact["text_value"],
                    "raw_value": selected_fact["raw_value"],
                    "decimals": selected_fact["decimals"],
                    "is_numeric": selected_fact["is_numeric"],
                }
            else:
                unified_record["unified_fields"][unified_field] = None

        unified_records.append(unified_record)

    return unified_records

## 16. Inspect Unified Financial Mapping on the Sample File

In [16]:
sample_unified_records = map_facts_to_unified_schema(
    raw_facts=sample_raw_facts,
    unified_mapping=UNIFIED_SCHEMA_MAPPING,
)

print("Total unified context records:", len(sample_unified_records))

for record in sample_unified_records[:3]:
    print(json.dumps(record, indent=2, ensure_ascii=False))
    print("-" * 100)

Total unified context records: 120
{
  "context_id": "OneI",
  "context": {
    "id": "OneI",
    "type": "instant",
    "start_date": null,
    "end_date": null,
    "instant_date": "2024-12-31",
    "dimension": null,
    "member": null
  },
  "taxonomy": "INSURANCE",
  "filing_metadata": {
    "company_name": "ICICI Lombard General Insurance Company Limited",
    "report_type": "Standalone",
    "currency": "INR",
    "rounding": null
  },
  "unified_fields": {
    "Operating_Revenue": null,
    "Total_Income": null,
    "Primary_Direct_Expense": null,
    "Employee_Expenses": null,
    "Net_Profit": null,
    "Basic_EPS": null
  }
}
----------------------------------------------------------------------------------------------------
{
  "context_id": "OneD",
  "context": {
    "id": "OneD",
    "type": "duration",
    "start_date": "2024-10-01",
    "end_date": "2024-12-31",
    "instant_date": null,
    "dimension": null,
    "member": null
  },
  "taxonomy": "INSURANCE",
  "filing

## 17. Define a Helper to Parse One XML File End to End

In [17]:
def parse_xbrl_file(file_path: Path) -> Dict[str, Any]:
    tree, root = load_xml_file(file_path)
    root = strip_namespaces(root)

    taxonomy = detect_taxonomy(root)
    filing_metadata = extract_filing_metadata(root)
    contexts = build_context_dictionary(root)

    raw_facts = extract_raw_facts(
        root=root,
        contexts=contexts,
        taxonomy=taxonomy,
        filing_metadata=filing_metadata,
    )

    unified_records = map_facts_to_unified_schema(
        raw_facts=raw_facts,
        unified_mapping=UNIFIED_SCHEMA_MAPPING,
    )

    return {
        "source_file": file_path.name,
        "taxonomy": taxonomy,
        "filing_metadata": filing_metadata,
        "context_count": len(contexts),
        "raw_fact_count": len(raw_facts),
        "unified_record_count": len(unified_records),
        "contexts": contexts,
        "raw_facts": raw_facts,
        "unified_records": unified_records,
    }

## 18. Test End-to-End Parsing on One Sample File

In [18]:
sample_parsed_output = parse_xbrl_file(sample_file)

print("Sample file parsed successfully.")
print("Source file:", sample_parsed_output["source_file"])
print("Detected taxonomy:", sample_parsed_output["taxonomy"])
print("Filing metadata:")
print(json.dumps(sample_parsed_output["filing_metadata"], indent=2, ensure_ascii=False))
print("Context count:", sample_parsed_output["context_count"])
print("Raw fact count:", sample_parsed_output["raw_fact_count"])
print("Unified record count:", sample_parsed_output["unified_record_count"])

Sample file parsed successfully.
Source file: GI_117338_1350247_17012025074725.xml
Detected taxonomy: INSURANCE
Filing metadata:
{
  "company_name": "ICICI Lombard General Insurance Company Limited",
  "report_type": "Standalone",
  "currency": "INR",
  "rounding": null
}
Context count: 128
Raw fact count: 339
Unified record count: 120


## 19. Parse All XML Files in the Input Folder

In [19]:
all_parsed_results = []
failed_files = []

for file_path in xml_files:
    print(f"Processing: {file_path.name}")

    try:
        parsed_output = parse_xbrl_file(file_path)
        all_parsed_results.append(parsed_output)
        print(
            f"  Success | Taxonomy: {parsed_output['taxonomy']} | "
            f"Contexts: {parsed_output['context_count']} | "
            f"Raw Facts: {parsed_output['raw_fact_count']} | "
            f"Unified Records: {parsed_output['unified_record_count']}"
        )
    except Exception as e:
        failed_files.append({
            "source_file": file_path.name,
            "error": str(e),
        })
        print(f"  Failed | Error: {e}")

Processing: GI_117338_1350247_17012025074725.xml
  Success | Taxonomy: INSURANCE | Contexts: 128 | Raw Facts: 339 | Unified Records: 120
Processing: INDAS_118123_1365717_30012025030002.xml
  Success | Taxonomy: COMMERCIAL_NBFC | Contexts: 11 | Raw Facts: 129 | Unified Records: 10
Processing: INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.xml
  Success | Taxonomy: BANK | Contexts: 57 | Raw Facts: 211 | Unified Records: 57
Processing: LI_117242_1346294_15012025050606.xml
  Success | Taxonomy: INSURANCE | Contexts: 130 | Raw Facts: 530 | Unified Records: 130
Processing: NBFC_INDAS_118212_1366575_30012025073535.xml
  Success | Taxonomy: COMMERCIAL_NBFC | Contexts: 48 | Raw Facts: 239 | Unified Records: 47
Processing: RELIANCE_1425445_25042025095716_WEB.xml
  Success | Taxonomy: BANK | Contexts: 54 | Raw Facts: 381 | Unified Records: 54
Processing: RELIANCE_1487275_18072025081722_WEB.xml
  Success | Taxonomy: COMMERCIAL_NBFC | Contexts: 36 | Raw Facts: 133 | Unified Records: 36
Proces

## 20. Review Batch Parsing Summary

In [20]:
taxonomy_counter = Counter(result["taxonomy"] for result in all_parsed_results)

total_contexts = sum(result["context_count"] for result in all_parsed_results)
total_raw_facts = sum(result["raw_fact_count"] for result in all_parsed_results)
total_unified_records = sum(result["unified_record_count"] for result in all_parsed_results)

print("Batch parsing summary")
print("-" * 80)
print("Total XML files found:", len(xml_files))
print("Successfully parsed files:", len(all_parsed_results))
print("Failed files:", len(failed_files))
print("Total contexts extracted:", total_contexts)
print("Total raw facts extracted:", total_raw_facts)
print("Total unified records created:", total_unified_records)

print("\nDetected taxonomy distribution:")
for taxonomy_name, count in taxonomy_counter.items():
    print(f"- {taxonomy_name}: {count}")

Batch parsing summary
--------------------------------------------------------------------------------
Total XML files found: 9
Successfully parsed files: 9
Failed files: 0
Total contexts extracted: 567
Total raw facts extracted: 2554
Total unified records created: 557

Detected taxonomy distribution:
- INSURANCE: 2
- COMMERCIAL_NBFC: 4
- BANK: 3


## 21. Inspect Failed Files, If Any

In [21]:
if SAVE_ONE_JSON_PER_XML:
    for parsed_result in all_parsed_results:
        output_file_name = Path(parsed_result["source_file"]).with_suffix(".json").name
        output_file_path = OUTPUT_FOLDER / output_file_name

        with output_file_path.open("w", encoding="utf-8") as file:
            json.dump(parsed_result, file, indent=2, ensure_ascii=False)

        print(f"Saved: {output_file_path.name}")

Saved: GI_117338_1350247_17012025074725.json
Saved: INDAS_118123_1365717_30012025030002.json
Saved: INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.json
Saved: LI_117242_1346294_15012025050606.json
Saved: NBFC_INDAS_118212_1366575_30012025073535.json
Saved: RELIANCE_1425445_25042025095716_WEB.json
Saved: RELIANCE_1487275_18072025081722_WEB.json
Saved: RELIANCE_1552809_17102025073643_WEB.json
Saved: RELIANCE_1602862_16012026073821_WEB.json


## 23. Save a Combined Batch Summary JSON File

In [24]:
batch_summary = []

for parsed_result in all_parsed_results:
    batch_summary.append({
        "source_file": parsed_result["source_file"],
        "taxonomy": parsed_result["taxonomy"],
        "company_name": parsed_result["filing_metadata"].get("company_name"),
        "report_type": parsed_result["filing_metadata"].get("report_type"),
        "currency": parsed_result["filing_metadata"].get("currency"),
        "rounding": parsed_result["filing_metadata"].get("rounding"),
        "context_count": parsed_result["context_count"],
        "raw_fact_count": parsed_result["raw_fact_count"],
        "unified_record_count": parsed_result["unified_record_count"],
    })

batch_summary_path = Path("../data/summary/batch_parsing_summary.json")

with open(batch_summary_path, "w", encoding="utf-8") as file:
    json.dump(batch_summary, file, indent=2, ensure_ascii=False)

print("Saved combined batch summary:", batch_summary_path.name)

Saved combined batch summary: batch_parsing_summary.json


## 24. Print Final Parser Completion Summary

In [25]:
print("Universal XBRL parser run completed.")
print("-" * 80)
print("Input folder:", INPUT_FOLDER.resolve())
print("Output folder:", OUTPUT_FOLDER.resolve())
print("Total XML files discovered:", len(xml_files))
print("Successfully parsed files:", len(all_parsed_results))
print("Failed files:", len(failed_files))
print("Detailed JSON outputs saved:", SAVE_ONE_JSON_PER_XML)
print("Batch summary file:", batch_summary_path.name)

Universal XBRL parser run completed.
--------------------------------------------------------------------------------
Input folder: C:\Users\Home\llmops-nse-rag\data\raw
Output folder: C:\Users\Home\llmops-nse-rag\data\processed
Total XML files discovered: 9
Successfully parsed files: 9
Failed files: 0
Detailed JSON outputs saved: True
Batch summary file: batch_parsing_summary.json
